# Signed Jumps 05 — does the jump hazard add to VD 19's model-versus-implied signal?

Protocol: `discussions/Signed_Jumps/011_claude_sj05_preregistration.md` (fixed before this run).

Reads VD 19 v0.1.1 rows (unchanged) → SJ 01 expected jumps from entry to the 08:00 expiry (model (e), no refit) →
$y_{rv,index}=a+\beta x+\gamma x_J$ at the 04:00 entry with VD 19's session bootstrap. Gate: γ lower bound > 0.

In [ ]:
import os,sys,json,time,importlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import signed_jumps as sj, signed_jumps_events as se, signed_jumps_vd19 as sv, model_vs_implied as mv
for m in (sj,se,sv):importlib.reload(m)
assert sv.VERSION=='SJ05-0.1.0' and mv.VERSION.startswith('VD19-0.1.1')
SYNTHETIC=os.getenv('SJ05_SYNTHETIC','0')=='1'
p=sj.Params();vp=sv.VdParams()
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='sj05-');OUT=Path(tmp.name)/'06'
else:
    OUT=ROOT/'notebooks'/'Signed_Jumps'/'06'
OUT.mkdir(parents=True,exist_ok=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
if not SYNTHETIC:
    chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_signed_jumps_vd19.py')],cwd=str(ROOT),env=env,capture_output=True,text=True)
    print(chk.stdout[-600:]);print(chk.stderr[-400:]);assert chk.returncode==0,'tests failed'
print(json.dumps(sv.registration(vp),indent=1))

## VD 19 rows (read only) and the SJ 01 jump hazard to expiry

In [ ]:
T0=time.time()
if SYNTHETIC:
    D=sv.synthetic_rows(n_sessions=25,gamma=0.5,seed=2)
else:
    rows=pd.read_csv(ROOT/'notebooks'/'Volatility_Dynamics'/'19'/'19_rows.csv')
    print('VD19 rows:',rows.shape,'| sessions',rows.session.min(),'->',rows.session.max());display(rows.groupby('entry_hour').size())
    BIN=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent/'Binance'
    S={A:sj.load_spot_1m(BIN/'basis'/'spot'/f'{A}USDT_spot_1m.parquet') for A in p.assets}
    assets,pan,_=sj.prepare(S,p)
    fits=se.fits_from_table(pd.read_csv(ROOT/'notebooks'/'Signed_Jumps'/'01'/'01_fits.csv'),pan,p,'e')
    orig=sv.origin_rows(pan,rows,vp)
    print('origins found:',int((orig>=0).sum()),'of',len(rows))
    lam,lam_b=sv.hazard_to_expiry(fits,assets,pan,p,orig,rows['h'].to_numpy(),'e')
    D=sv.attach(rows,lam,lam_b)
    D[['session','underlying','entry_hour','h','x','Lambda','Lambda_base','xJ','y_rv_index','y_rv','y','R_net']].to_csv(OUT/'06_rows.csv',index=False)
    display(D.groupby('entry_hour')[['Lambda','Lambda_base','xJ','x']].describe().T.round(3))
print(f'{time.time()-T0:.0f}s')

## Information: $y=a+\beta x+\gamma x_J$ (and $x_J$ alone)

In [ ]:
info=sv.information(D,mv,vp);info.to_csv(OUT/'06_information.csv',index=False)
cols=[c for c in ['entry_hour','outcome','model','n','sessions','corr_x_xJ','x','x_lo','x_hi','xJ','xJ_lo','xJ_hi'] if c in info]
display(info[cols].round(3))

## Economics (descriptive): VD 19-selected straddles split by jump hazard

In [ ]:
if 'R_net' in D and D['R_net'].notna().any():
    eco=sv.economics(D,mv,vp);eco.to_csv(OUT/'06_economics.csv',index=False);display(eco.round(3))

## Decision

In [ ]:
dec=sv.decide(info,vp);dec['synthetic']=SYNTHETIC
print(json.dumps(dec,indent=1,default=float))
if not SYNTHETIC:
    (OUT/'06_decision.json').write_text(json.dumps(dec,indent=1,default=float))
    (OUT/'06_manifest.json').write_text(json.dumps({'registration':sv.registration(vp),'runtime_s':time.time()-T0},indent=1,default=str))